# URL 위험 분류 Transformer 파인튜닝

`dataset/open/train.csv`와 `dataset/archive/malicious_phish.csv`를 이용해 문자 단위 사전학습 모델을 파인튜닝합니다. 기존 `url-risk-model.ipynb`의 선형 모델은 비교 기준으로 유지합니다.

진행 순서: 환경 준비 → GPU 확인 → 데이터 로드·기록 → URL 정제·라벨 충돌 검사 → 도메인 그룹 분할 → 모델 로드·토큰화 → 학습 → 평가·저장

## 1. 환경 준비

Colab에서는 먼저 런타임 유형을 GPU로 바꾼 뒤 실행합니다. Google Drive는 사용하지 않습니다. 로컬에서는 프로젝트 루트에서 이 노트북을 여세요. 패키지를 처음 설치한 직후 import 오류가 나면 커널을 한 번 재시작합니다.

In [11]:
%pip install -q -r training/requirements-transformer.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
from pathlib import Path
import json
import os
import platform
import random
import re
import sys
import warnings

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from transformers import set_seed

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

SEED = 42
MODEL_NAME = 'google/canine-s'
MAX_LENGTH = 256
RUN_NAME = 'canine-url-v1'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

def find_project_dir(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'dataset' / 'open' / 'train.csv').is_file():
            return candidate
    colab_candidate = Path('/content/smishing-checker')
    if (colab_candidate / 'dataset' / 'open' / 'train.csv').is_file():
        return colab_candidate
    raise FileNotFoundError('dataset/open/train.csv가 있는 저장소 루트에서 실행하세요.')

PROJECT_DIR = find_project_dir(Path.cwd())
DATA_DIR = PROJECT_DIR / 'dataset'
OPEN_TRAIN_PATH = DATA_DIR / 'open' / 'train.csv'
OPEN_TEST_PATH = DATA_DIR / 'open' / 'test.csv'
ARCHIVE_PATH = DATA_DIR / 'archive' / 'malicious_phish.csv'
RUN_DIR = PROJECT_DIR / 'experiments' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('OS:', platform.platform())
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('scikit-learn:', sklearn.__version__)
print('프로젝트:', PROJECT_DIR)
print('실험 저장 경로:', RUN_DIR)

Python: 3.12.10
OS: Windows-11-10.0.26200-SP0
PyTorch: 2.11.0+cu128
Transformers: 4.57.1
scikit-learn: 1.6.1
프로젝트: C:\Users\SSAFY\smishing-checker
실험 저장 경로: C:\Users\SSAFY\smishing-checker\experiments\canine-url-v1


## 2. GPU 연결 확인

코드가 GPU를 직접 연결할 수는 없습니다. Colab 메뉴의 `런타임 → 런타임 유형 변경 → GPU`에서 연결한 뒤 아래 셀로 확인합니다. GPU가 없으면 즉시 중단해 CPU로 실수로 학습하는 일을 막습니다.

In [13]:
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 연결되지 않았습니다. Colab 런타임 또는 로컬 CUDA 환경을 확인하세요.')

DEVICE = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
COMPUTE_CAPABILITY = torch.cuda.get_device_capability(0)
USE_BF16 = bool(torch.cuda.is_bf16_supported())
USE_FP16 = not USE_BF16

print('GPU:', GPU_NAME)
print(f'GPU 메모리: {GPU_MEMORY_GB:.1f} GB')
print('Compute capability:', COMPUTE_CAPABILITY)
print('정밀도:', 'bf16' if USE_BF16 else 'fp16')
assert GPU_MEMORY_GB >= 8, '8GB 미만 GPU에서는 배치 크기와 MAX_LENGTH를 더 줄여야 합니다.'

GPU: NVIDIA GeForce RTX 5060 Ti
GPU 메모리: 15.9 GB
Compute capability: (12, 0)
정밀도: bf16


## 3. 데이터 로드 및 출처 기록

전체 CSV를 메모리에 합치지 않고 청크 단위로 읽습니다. 스키마, 행 수, URL 결측치, 라벨 분포와 SHA-256을 기록해 실험 데이터를 재현할 수 있게 합니다. `archive`의 `benign`은 0, `phishing/defacement/malware`는 1로 통일합니다.

In [14]:
from training.data_manifest import build_data_manifest

MANIFEST_PATH = RUN_DIR / 'data_manifest.json'
data_manifest = build_data_manifest(
    project_dir=PROJECT_DIR,
    output_path=MANIFEST_PATH,
    chunk_size=250_000,
)

summary_rows = []
for item in data_manifest['files']:
    labels = item['binary_label_counts'] or {}
    summary_rows.append({
        'source': item['source'],
        'rows': item['rows'],
        'normal': labels.get('normal'),
        'malicious': labels.get('malicious'),
        'missing_urls': item['missing_urls'],
        'empty_urls': item['empty_urls'],
        'size_mb': round(item['size_bytes'] / (1024 ** 2), 1),
        'sha256_short': item['sha256'][:12],
    })

display(pd.DataFrame(summary_rows))
print('통합 라벨 분포:', data_manifest['combined_labeled'])
print('기록 파일:', MANIFEST_PATH)
print('open_test.csv에는 정답 label이 없어 최종 확률 생성에만 사용합니다.')

,source,rows,normal,malicious,missing_urls,empty_urls,size_mb,sha256_short
0,open_train,6995056,5430159.0,1564897.0,0,0,296.3,a7cc3e326085
1,archive,651191,428103.0,223088.0,0,0,43.5,d83ce942075d
2,open_test,1747689,NaN,NaN,0,0,69.0,96f3a9113ddb


통합 라벨 분포: {'rows': 7646247, 'normal': 5858262, 'malicious': 1787985, 'malicious_ratio': 0.2338382477050506}
기록 파일: C:\Users\SSAFY\smishing-checker\experiments\canine-url-v1\data_manifest.json
open_test.csv에는 정답 label이 없어 최종 확률 생성에만 사용합니다.


In [15]:
assert all(item['missing_urls'] == 0 for item in data_manifest['files'])
assert all(item['empty_urls'] == 0 for item in data_manifest['files'])
assert data_manifest['combined_labeled']['rows'] > 0
assert 0 < data_manifest['combined_labeled']['malicious_ratio'] < 1
print('데이터 스키마·URL 결측·통합 라벨 검사 통과')

데이터 스키마·URL 결측·통합 라벨 검사 통과


## 4. URL 정제 및 라벨 충돌 검사

전체 764만 행은 DuckDB로 디스크 기반 처리합니다. `[.]` 복원, 소문자화와 스킴 통일 후 동일 URL의 중복을 제거합니다. 같은 URL에 0과 1이 함께 붙은 충돌 데이터는 학습에서 제외하고 별도 CSV로 기록합니다. Transformer 학습 시간 때문에 각 클래스의 학습 후보 수는 파라미터로 제한하며, 원본 데이터는 수정하지 않습니다.

In [16]:
import duckdb

MAX_ROWS_BY_LABEL = {0: 300_000, 1: 300_000}
DUCKDB_PATH = RUN_DIR / 'preprocessing.duckdb'
CONFLICT_PATH = RUN_DIR / 'label_conflicts.csv'
SAMPLED_PATH = RUN_DIR / 'sampled_urls.parquet'

def sql_path(path: Path) -> str:
    return path.resolve().as_posix().replace("'", "''")

connection = duckdb.connect(str(DUCKDB_PATH))
connection.execute("PRAGMA threads=4")
connection.execute("PRAGMA memory_limit='6GB'")
connection.execute(f"""
CREATE OR REPLACE TABLE normalized_rows AS
WITH combined AS (
    SELECT URL AS raw_url, CAST(label AS UTINYINT) AS label, 'open' AS source
    FROM read_csv('{sql_path(OPEN_TRAIN_PATH)}', header=true, all_varchar=true)
    UNION ALL
    SELECT url AS raw_url,
           CASE lower(trim(type))
               WHEN 'benign' THEN 0
               WHEN 'phishing' THEN 1
               WHEN 'defacement' THEN 1
               WHEN 'malware' THEN 1
           END AS label,
           'archive' AS source
    FROM read_csv('{sql_path(ARCHIVE_PATH)}', header=true, all_varchar=true)
), cleaned AS (
    SELECT trim(replace(lower(raw_url), '[.]', '.')) AS value, label, source
    FROM combined
)
SELECT CASE
           WHEN starts_with(value, 'hxxps://') THEN 'https://' || substr(value, 9)
           WHEN starts_with(value, 'hxxp://') THEN 'http://' || substr(value, 8)
           WHEN regexp_matches(value, '^https?://') THEN value
           ELSE 'https://' || value
       END AS normalized_url,
       label, source
FROM cleaned
WHERE value IS NOT NULL AND value <> '' AND label IN (0, 1)
""")

connection.execute(f"""
COPY (
    SELECT normalized_url, count(*) AS row_count,
           min(label) AS min_label, max(label) AS max_label
    FROM normalized_rows
    GROUP BY normalized_url
    HAVING min(label) <> max(label)
) TO '{sql_path(CONFLICT_PATH)}' (HEADER, DELIMITER ',')
""")

connection.execute("""
CREATE OR REPLACE TABLE clean_urls AS
SELECT normalized_url, min(label) AS label, string_agg(DISTINCT source, ',') AS source
FROM normalized_rows
GROUP BY normalized_url
HAVING min(label) = max(label)
""")

connection.execute(f"""
CREATE OR REPLACE TABLE sampled_urls AS
WITH ranked AS (
    SELECT *, row_number() OVER (PARTITION BY label ORDER BY hash(normalized_url)) AS rank
    FROM clean_urls
)
SELECT normalized_url, CAST(label AS TINYINT) AS label, source
FROM ranked
WHERE (label = 0 AND rank <= {MAX_ROWS_BY_LABEL[0]})
   OR (label = 1 AND rank <= {MAX_ROWS_BY_LABEL[1]})
""")
connection.execute(f"COPY sampled_urls TO '{sql_path(SAMPLED_PATH)}' (FORMAT PARQUET, COMPRESSION ZSTD)")

preprocessing_stats = {
    'raw_rows': connection.execute('SELECT count(*) FROM normalized_rows').fetchone()[0],
    'unique_non_conflicting_urls': connection.execute('SELECT count(*) FROM clean_urls').fetchone()[0],
    'conflicting_urls': connection.execute(
        'SELECT count(*) FROM (SELECT 1 FROM normalized_rows GROUP BY normalized_url HAVING min(label) <> max(label))'
    ).fetchone()[0],
    'sampled_label_counts': dict(
        connection.execute('SELECT label, count(*) FROM sampled_urls GROUP BY label ORDER BY label').fetchall()
    ),
}
connection.close()
(RUN_DIR / 'preprocessing_stats.json').write_text(
    json.dumps(preprocessing_stats, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(preprocessing_stats, ensure_ascii=False, indent=2))
print('충돌 보고서:', CONFLICT_PATH)
print('학습 후보:', SAMPLED_PATH)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "raw_rows": 7646247,
  "unique_non_conflicting_urls": 7450009,
  "conflicting_urls": 79471,
  "sampled_label_counts": {
    "0": 300000,
    "1": 300000
  }
}
충돌 보고서: C:\Users\SSAFY\smishing-checker\experiments\canine-url-v1\label_conflicts.csv
학습 후보: C:\Users\SSAFY\smishing-checker\experiments\canine-url-v1\sampled_urls.parquet


## 5. 등록 도메인 그룹 기준 분할

같은 등록 도메인의 하위 URL이 학습과 평가에 동시에 들어가면 성능이 과대평가됩니다. 오프라인 Public Suffix 목록으로 등록 도메인을 구한 뒤 안정적인 해시를 사용해 train 80%, validation 10%, internal_test 10%로 분리합니다. 공식 대회 test는 이 분할에 포함하지 않습니다.

In [17]:
import hashlib
from urllib.parse import urlsplit
import tldextract

extract_domain = tldextract.TLDExtract(suffix_list_urls=())
sampled_df = pd.read_parquet(SAMPLED_PATH)

def safe_hostname(url: str) -> str:
    try:
        return (urlsplit(url).hostname or '').lower().strip('.')
    except ValueError:
        # 깨진 대괄호 등을 IPv6 표기로 오인한 경우 authority에서 직접 복구합니다.
        authority_match = re.match(r'^(?:[a-z][a-z0-9+.-]*://)?([^/?#]+)', str(url), re.I)
        if not authority_match:
            return ''
        authority = authority_match.group(1).rsplit('@', 1)[-1].strip().lower()
        authority = authority.replace('[', '').replace(']', '')
        if authority.count(':') == 1:
            authority = authority.split(':', 1)[0]
        return authority.strip('.')

def registered_domain(url: str) -> str:
    host = safe_hostname(url)
    if not host:
        # 호스트를 전혀 복구하지 못하면 URL별 안정 해시 그룹으로 분리합니다.
        digest = hashlib.blake2b(str(url).encode('utf-8'), digest_size=8).hexdigest()
        return f'invalid-{digest}'
    extracted = extract_domain(host)
    return extracted.top_domain_under_public_suffix or host

def group_split(domain: str) -> str:
    bucket = int.from_bytes(
        hashlib.blake2b(domain.encode('utf-8'), digest_size=8).digest(), 'big'
    ) % 10_000
    if bucket < 1_000:
        return 'internal_test'
    if bucket < 2_000:
        return 'validation'
    return 'train'

sampled_df['domain_group'] = sampled_df['normalized_url'].map(registered_domain)
sampled_df['split'] = sampled_df['domain_group'].map(group_split)

split_paths = {}
for split_name in ('train', 'validation', 'internal_test'):
    split_path = RUN_DIR / f'{split_name}.parquet'
    sampled_df.loc[sampled_df['split'].eq(split_name), [
        'normalized_url', 'label', 'source', 'domain_group'
    ]].to_parquet(split_path, index=False)
    split_paths[split_name] = split_path

domain_sets = {
    name: set(sampled_df.loc[sampled_df['split'].eq(name), 'domain_group'])
    for name in split_paths
}
assert domain_sets['train'].isdisjoint(domain_sets['validation'])
assert domain_sets['train'].isdisjoint(domain_sets['internal_test'])
assert domain_sets['validation'].isdisjoint(domain_sets['internal_test'])

split_summary = pd.crosstab(sampled_df['split'], sampled_df['label']).rename(
    columns={0: 'normal', 1: 'malicious'}
)
display(split_summary)
print('도메인 그룹 누수 검사 통과')

label,normal,malicious
split,,
internal_test,30788,31385
train,236677,244041
validation,32535,24574


도메인 그룹 누수 검사 통과


## 6. CANINE 모델 로드 및 문자 단위 토큰화

CANINE은 URL의 점, 슬래시, 하이픈과 숫자를 문자 단위로 처리합니다. 최대 길이를 넘는 URL만 자르고, 배치 안에서 가장 긴 입력에 맞춰 동적 패딩합니다.

In [18]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset(
    'parquet',
    data_files={name: str(path) for name, path in split_paths.items()},
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch['normalized_url'],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    batch_size=2_000,
    remove_columns=['normalized_url', 'source', 'domain_group'],
    desc='URL 문자 토큰화',
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'NORMAL', 1: 'MALICIOUS'},
    label2id={'NORMAL': 0, 'MALICIOUS': 1},
)
print(raw_datasets)
print('학습 가능 파라미터:', f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating internal_test split: 0 examples [00:00, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/854 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

...

URL 문자 토큰화:   0%|          | 0/62173 [00:00<?, ? examples/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/528M [00:00<?, ?B/s]

Some weights of CanineForSequenceClassification were not initialized from the model checkpoint at google/canine-s and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DatasetDict({
    train: Dataset({
        features: ['normalized_url', 'label', 'source', 'domain_group'],
        num_rows: 480718
    })
    validation: Dataset({
        features: ['normalized_url', 'label', 'source', 'domain_group'],
        num_rows: 57109
    })
    internal_test: Dataset({
        features: ['normalized_url', 'label', 'source', 'domain_group'],
        num_rows: 62173
    })
})
학습 가능 파라미터: 132,084,482


## 7. 평가 지표와 파인튜닝

ROC-AUC뿐 아니라 불균형 데이터에 더 민감한 PR-AUC, Precision, Recall, F1과 정상 URL 오탐률을 함께 기록합니다. validation의 PR-AUC가 가장 높은 체크포인트를 선택합니다.

In [21]:
from sklearn.metrics import (
    average_precision_score, confusion_matrix, precision_recall_fscore_support,
    roc_auc_score, roc_curve,
)
from torch.nn import CrossEntropyLoss
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments

def softmax_probabilities(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits[:, 1] / exp_logits.sum(axis=1)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    probabilities = softmax_probabilities(logits)
    predictions = (probabilities >= 0.5).astype(np.int8)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        'roc_auc': roc_auc_score(labels, probabilities),
        'average_precision': average_precision_score(labels, probabilities),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'normal_false_positive_rate': fp / max(1, fp + tn),
    }

train_labels = np.asarray(raw_datasets['train']['label'], dtype=np.int64)
class_counts = np.bincount(train_labels, minlength=2)
class_weights = torch.tensor(
    class_counts.sum() / (2 * np.maximum(class_counts, 1)), dtype=torch.float32
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))(
            outputs.logits, labels
        )
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=str(RUN_DIR / 'checkpoints'),
    num_train_epochs=1,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    warmup_ratio=0.10,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='average_precision',
    greater_is_better=True,
    save_total_limit=2,
    bf16=USE_BF16,
    fp16=USE_FP16,
    tf32=True,
    dataloader_num_workers=2,
    report_to='none',
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [22]:
train_result = trainer.train()
trainer.log_metrics('train', train_result.metrics)
trainer.save_metrics('train', train_result.metrics)
trainer.save_state()

Epoch,Training Loss,Validation Loss,Roc Auc,Average Precision,Precision,Recall,F1,Normal False Positive Rate
1,0.222200,0.251578,0.952730,0.954979,0.955568,0.827908,0.887169,0.029076


***** train metrics *****
  epoch                    =        1.0
  total_flos               = 29134299GF
  train_loss               =     0.2598
  train_runtime            = 1:00:54.45
  train_samples_per_second =    131.543
  train_steps_per_second   =      4.111


## 8. 내부 테스트 평가와 모델 저장

validation에서 정상 오탐률 5% 이하를 만족하면서 Recall이 최대가 되는 임계값을 선택합니다. 임계값 선택에 사용하지 않은 internal_test에서 최종 성능을 보고한 뒤 모델과 임계값을 저장합니다.

In [23]:
from sklearn.metrics import classification_report

MAX_NORMAL_FPR = 0.05
validation_output = trainer.predict(tokenized_datasets['validation'])
validation_probabilities = softmax_probabilities(validation_output.predictions)
fpr, tpr, thresholds = roc_curve(validation_output.label_ids, validation_probabilities)
candidates = np.flatnonzero((fpr <= MAX_NORMAL_FPR) & np.isfinite(thresholds))
if candidates.size == 0:
    raise RuntimeError('요구한 정상 오탐률을 만족하는 임계값이 없습니다.')
best_recall = tpr[candidates].max()
best_candidates = candidates[tpr[candidates] == best_recall]
decision_threshold = float(thresholds[best_candidates].max())

internal_test_output = trainer.predict(tokenized_datasets['internal_test'])
internal_test_probabilities = softmax_probabilities(internal_test_output.predictions)
internal_test_labels = internal_test_output.label_ids
internal_test_predictions = (internal_test_probabilities >= decision_threshold).astype(np.int8)
tn, fp, fn, tp = confusion_matrix(
    internal_test_labels, internal_test_predictions, labels=[0, 1]
).ravel()

final_metrics = {
    'threshold': decision_threshold,
    'threshold_policy': {'max_normal_false_positive_rate': MAX_NORMAL_FPR},
    'roc_auc': float(roc_auc_score(internal_test_labels, internal_test_probabilities)),
    'average_precision': float(average_precision_score(internal_test_labels, internal_test_probabilities)),
    'normal_false_positive_rate': float(fp / max(1, fp + tn)),
    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    'classification_report': classification_report(
        internal_test_labels, internal_test_predictions,
        target_names=['NORMAL', 'MALICIOUS'], output_dict=True, zero_division=0,
    ),
}

MODEL_DIR = RUN_DIR / 'model'
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
(RUN_DIR / 'internal_test_metrics.json').write_text(
    json.dumps(final_metrics, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(final_metrics, ensure_ascii=False, indent=2))
print('저장 모델:', MODEL_DIR)

{
  "threshold": 0.32082128524780273,
  "threshold_policy": {
    "max_normal_false_positive_rate": 0.05
  },
  "roc_auc": 0.9621501213238736,
  "average_precision": 0.9707005234944333,
  "normal_false_positive_rate": 0.048817721190074055,
  "confusion_matrix": {
    "tn": 29285,
    "fp": 1503,
    "fn": 3787,
    "tp": 27598
  },
  "classification_report": {
    "NORMAL": {
      "precision": 0.885492259313014,
      "recall": 0.9511822788099259,
      "f1-score": 0.9171625430629502,
      "support": 30788.0
    },
    "MALICIOUS": {
      "precision": 0.9483522902993025,
      "recall": 0.8793372630237375,
      "f1-score": 0.9125417451972357,
      "support": 31385.0
    },
    "accuracy": 0.9149148344136522,
    "macro avg": {
      "precision": 0.9169222748061583,
      "recall": 0.9152597709168317,
      "f1-score": 0.914852144130093,
      "support": 62173.0
    },
    "weighted avg": {
      "precision": 0.917224073323994,
      "recall": 0.9149148344136522,
      "f1-score": 

## 9. 대회 test 추론 및 제출 CSV 생성

`dataset/open/test.csv`의 순서를 유지해 모든 URL의 악성 확률을 계산합니다. `sample_submission.csv`의 ID와 매 청크 대조하고 `ID,probability` 두 열만 저장합니다. 대회 평가지표용 제출에는 내부 판정 임계값을 적용하지 않고 0~1 확률을 그대로 사용합니다.

In [24]:
import csv
from itertools import zip_longest

SAMPLE_SUBMISSION_PATH = DATA_DIR / 'open' / 'sample_submission.csv'
SUBMISSION_PATH = RUN_DIR / 'submission.csv'
TEST_CHUNK_SIZE = 50_000
INFERENCE_BATCH_SIZE = 64

def normalize_test_url(value: str) -> str:
    url = str(value).strip().lower().replace('[.]', '.')
    if url.startswith('hxxps://'):
        return 'https://' + url[8:]
    if url.startswith('hxxp://'):
        return 'http://' + url[7:]
    return url if re.match(r'^https?://', url) else 'https://' + url

model.to(DEVICE)
model.eval()
written_rows = 0
test_chunks = pd.read_csv(OPEN_TEST_PATH, dtype='string', chunksize=TEST_CHUNK_SIZE)
sample_chunks = pd.read_csv(
    SAMPLE_SUBMISSION_PATH, usecols=['ID'], dtype='string', chunksize=TEST_CHUNK_SIZE
)

with SUBMISSION_PATH.open('w', encoding='utf-8', newline='') as output_file:
    writer = csv.writer(output_file)
    writer.writerow(['ID', 'probability'])
    for test_chunk, sample_chunk in zip_longest(test_chunks, sample_chunks):
        if test_chunk is None or sample_chunk is None:
            raise ValueError('test.csv와 sample_submission.csv의 청크 수가 다릅니다.')
        if not test_chunk['ID'].reset_index(drop=True).equals(sample_chunk['ID'].reset_index(drop=True)):
            raise ValueError('test.csv와 sample_submission.csv의 ID 순서가 다릅니다.')

        normalized_urls = [normalize_test_url(value) for value in test_chunk['URL']]
        chunk_probabilities = []
        for start in range(0, len(normalized_urls), INFERENCE_BATCH_SIZE):
            encoded = tokenizer(
                normalized_urls[start:start + INFERENCE_BATCH_SIZE],
                truncation=True, padding=True, max_length=MAX_LENGTH,
                return_tensors='pt',
            ).to(DEVICE)
            with torch.inference_mode(), torch.autocast(
                device_type='cuda', dtype=torch.bfloat16 if USE_BF16 else torch.float16
            ):
                logits = model(**encoded).logits.float().cpu().numpy()
            chunk_probabilities.extend(softmax_probabilities(logits).tolist())

        writer.writerows(
            (sample_id, f'{probability:.8f}')
            for sample_id, probability in zip(sample_chunk['ID'], chunk_probabilities)
        )
        written_rows += len(test_chunk)
        print(f'제출 추론: {written_rows:,}행 완료', end='\r')

print(f'\n제출 파일 생성: {SUBMISSION_PATH}')

제출 추론: 1,747,689행 완료
제출 파일 생성: C:\Users\SSAFY\smishing-checker\experiments\canine-url-v1\submission.csv


In [25]:
submission_df = pd.read_csv(SUBMISSION_PATH, dtype={'ID': 'string'})
sample_ids = pd.read_csv(SAMPLE_SUBMISSION_PATH, usecols=['ID'], dtype={'ID': 'string'})
submission_probabilities = pd.to_numeric(submission_df['probability'], errors='coerce')

assert submission_df.columns.tolist() == ['ID', 'probability']
assert len(submission_df) == len(sample_ids)
assert submission_df['ID'].equals(sample_ids['ID'])
assert submission_df['ID'].is_unique
assert submission_probabilities.notna().all()
assert submission_probabilities.between(0, 1).all()

print('제출 파일 검증 통과')
print('행 수:', f'{len(submission_df):,}')
print('확률 범위:', float(submission_probabilities.min()), '~', float(submission_probabilities.max()))
display(submission_df.head())

제출 파일 검증 통과
행 수: 1,747,689
확률 범위: 0.00280093 ~ 0.99865413


,ID,probability
0,TEST_0000000,0.093347
1,TEST_0000001,0.078642
2,TEST_0000002,0.027796
3,TEST_0000003,0.003124
4,TEST_0000004,0.070560


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 저장된 파일에서 새로 로딩
tokenizer_check = AutoTokenizer.from_pretrained(
    str(MODEL_DIR), local_files_only=True
)
model_check = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR), local_files_only=True
)
model_check.eval()

inputs = tokenizer_check(
    "https://www.kbstar.com/",
    return_tensors="pt",
    truncation=True,
    max_length=256,
)

with torch.inference_mode():
    probabilities = torch.softmax(model_check(**inputs).logits, dim=-1)

print("모델 경로:", MODEL_DIR)
print("라벨 설정:", model_check.config.id2label)
print("클래스별 확률:", probabilities.tolist())
print("모델 로딩 및 추론 성공")